In [64]:
# qcqp_regression_aligned_keep_unbounded.py
#
# POLICY IN THIS VERSION
# ------------------------------------------------------------
# 1) Unified statuses across pipelines:
#       OPTIMAL_STATUSES
#       UNBOUNDED_STATUSES
#       EXCLUDE_STATUSES
#
# 2) Keep unbounded rows in the regression dataset:
#       SDP unbounded + LP finite  -> y = +1.0
#       LP  unbounded + SDP finite -> y = -1.0
#       finite / finite            -> y = fixed additive delta
#       both failed / excluded     -> removed (no usable target)
#
# 3) MSE / MAE / Pearson are computed on FINITE rows only.
#    Unbounded rows stay in training and prediction, but do NOT enter
#    finite-gap error metrics.
#
# 4) LP-biased decision rule everywhere:
#       choose LP iff delta > -TAU_SIGN
#    This is used in:
#       - r_accuracy
#       - sign_accuracy_tau
#       - sign confusion matrix
#
# 5) Sample weights are unified and logical everywhere:
#       w_i = (1 - WEIGHT_ALPHA) + WEIGHT_ALPHA * |y_i|
#    Since unbounded rows have y = ±1, they receive weight 1.0.
#
# 6) FIX FOR YOUR CRASH:
#    Targets are built BEFORE the train/test split, so X and y always stay aligned.

import re
from typing import List, Optional, Tuple, Dict
import pandas as pd
import numpy as np
from pathlib import Path
from pandas.api.types import CategoricalDtype
from datetime import datetime

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available. Install with: pip install xgboost")


# ============================================================
# FEATURE FILE CONFIG
# ============================================================
FEATURE_FILE = "selected_featuresregfdr.txt"

# ============================================================
# CONFIG
SETUP = "DI"

TRAIN_FEATURES = ["features_5_10_10a_DI_synth.xlsx"]
TRAIN_OBJECTS  = ["instancesobj_5_10_10a.csv"]

TEST_FEATURES = ["features_10_2a_DI_synth.xlsx"]
TEST_OBJECTS  = ["instancesobj_10_2a.csv"]

USE_HOLDOUT_FROM_TRAIN = (len(TEST_FEATURES) == 0)

KEY_COL = "id_key"



USE_HOLDOUT_FROM_TRAIN = (len(TEST_FEATURES) == 0)
KEY_COL = "id_key"

# ============================================================
# REGRESSION MODE
# ============================================================
MODE = "pairwise"  # "pairwise" or "multi"
PAIRWISE_RELAXATIONS = ("lp", "sdp_prime")
MULTI_RELAXATIONS    = ["lp", "sdp", "sdp_prime", "sdp_rlt"]
MULTI_BASELINE       = "lp"
ALL_RELAXATIONS      = ["lp", "sdp", "sdp_prime", "sdp_rlt", "sdpp"]

# ============================================================
# TARGET SHAPING + EPSILON
# ============================================================
EPSILON_GAP     = 1e-3
EPSILON_METHOD  = "constant"
DELTA_TRANSFORM = "none"
TAU_SIGN        = 0.01

# ============================================================
# SAMPLE WEIGHTS
# ============================================================
USE_SAMPLE_WEIGHTS = True
WEIGHT_ALPHA = 0.30  # w = (1-alpha) + alpha*|y|

# ============================================================
# STATUS SETS
# ============================================================
OPTIMAL_STATUSES = frozenset({
    "optimal",
    "optimal_inaccurate",
})

UNBOUNDED_STATUSES = frozenset({
    "unbounded",
    "unbounded_inaccurate",
    "unbounded_uncertified",
})

EXCLUDE_STATUSES = frozenset({
    "infeasible",
    "infeasible_inaccurate",
    "infeasible_or_unbounded",
    "primal_infeasible",
    "dual_infeasible_cert",
    "unknown",
    "numeric_error",
    "time_limit",
    "iteration_limit",
    "node_limit",
    "interrupted",
    "memory_limit",
})

RANDOM_STATE = 42
TEST_SIZE    = 0.2

OUTPUT_ROOT = Path("reg_outputs_txt_features_gnn_synth_missing")
OUTPUT_ROOT.mkdir(exist_ok=True, parents=True)

# ============================================================
# OBJECTIVE COLUMN NORMALIZATION
# ============================================================
OBJ_RENAME_MAP = {
    "sdpp_obj":        "sdp_prime_obj",
    "sdpp_status":     "sdp_prime_status",
    "sdprlt_obj":      "sdp_rlt_obj",
    "sdprlt_status":   "sdp_rlt_status",
    "sdpprime_obj":    "sdp_prime_obj",
    "sdpprime_status": "sdp_prime_status",
}

def normalize_objective_columns(Yf):
    Yf = Yf.copy()
    cols = {c: OBJ_RENAME_MAP[c] for c in Yf.columns if c in OBJ_RENAME_MAP}
    if cols:
        Yf = Yf.rename(columns=cols)
    return Yf

# ============================================================
# HELPERS
# ============================================================
def _ts():
    return datetime.now().strftime("%H:%M:%S")

def infer_tag(path):
    stem = Path(path).stem
    m = re.search(r"(\d+_\d+)(?:$|_)", stem)
    if m:
        return m.group(1)
    parts = stem.split("_", 1)
    return parts[1] if len(parts) > 1 else stem

def run_tag_from_lists(fpaths):
    tags = sorted({infer_tag(p) for p in fpaths})
    return "+".join(tags) if tags else "unk"

def read_any(path):
    p = Path(path)
    if p.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(p)
    if p.suffix.lower() == ".csv":
        return pd.read_csv(p)
    raise ValueError(f"Unsupported: {p.suffix}")

def _norm_status(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return str(x).strip().lower()

def _norm_status_series(s):
    return s.astype(str).str.strip().str.lower()

# ============================================================
# DETERMINISTIC SPLIT
# ============================================================
def make_deterministic_split(keys: pd.Series, test_frac: float = 0.2):
    h = pd.util.hash_pandas_object(keys.astype(str), index=False).to_numpy(dtype=np.uint64)
    b = (h % 100).astype(np.int64)
    cut = int((1.0 - test_frac) * 100)
    return np.where(b < cut)[0], np.where(b >= cut)[0]

# ============================================================
# FEATURE FILE
# ============================================================
def load_features_from_txt(feature_file):
    if feature_file is None:
        print(f"[{_ts()}] [features] No file - using ALL features", flush=True)
        return None
    fpath = Path(feature_file)
    if not fpath.exists():
        print(f"[{_ts()}] [features] Not found: {feature_file} - using ALL", flush=True)
        return None
    try:
        with open(fpath, "r") as f:
            feats = [line.strip() for line in f if line.strip() and not line.strip().startswith("#")]
        print(f"[{_ts()}] [features] Loaded {len(feats)} features from {feature_file}", flush=True)
        for i, feat in enumerate(feats[:10], 1):
            print(f"[{_ts()}]   {i:2d}. {feat}", flush=True)
        if len(feats) > 10:
            print(f"[{_ts()}]   ... and {len(feats)-10} more", flush=True)
        return feats
    except Exception as e:
        print(f"[{_ts()}] [features] Error: {e} - using ALL", flush=True)
        return None

def filter_to_selected_features(X, selected_features):
    if selected_features is None:
        print(f"[{_ts()}] [features] Using ALL {X.shape[1]} features", flush=True)
        return X
    available = [f for f in selected_features if f in X.columns]
    missing   = [f for f in selected_features if f not in X.columns]
    if missing:
        print(f"[{_ts()}] [features] {len(missing)} not found", flush=True)
        for f in missing[:5]:
            print(f"[{_ts()}]   - {f}", flush=True)
        if len(missing) > 5:
            print(f"[{_ts()}]   ... and {len(missing)-5} more", flush=True)
    if len(available) == 0:
        print(f"[{_ts()}] [features] NO features found - using ALL", flush=True)
        return X
    print(f"[{_ts()}] [features] Using {len(available)}/{len(selected_features)} features", flush=True)
    return X[available]

# ============================================================
# MERGE
# ============================================================
def merge_features_objectives(Xf, Yf, prefer="id_global", required_obj_cols=None):
    Xf = Xf.copy()
    Yf = Yf.copy()

    if ("id_global" not in Xf.columns) and ("id" in Xf.columns) and ("id_global" in Yf.columns):
        if Xf["id"].astype(str).str.contains("__").any():
            Xf["id_global"] = Xf["id"].astype(str)

    if ("id_global" not in Yf.columns) and ("id" in Yf.columns) and ("id_global" in Xf.columns):
        if Yf["id"].astype(str).str.contains("__").any():
            Yf["id_global"] = Yf["id"].astype(str)

    key = None
    for k in [prefer, "id_global", "id"]:
        if (k in Xf.columns) and (k in Yf.columns):
            key = k
            break

    if key is None:
        raise ValueError(f"No common merge key.\nX cols: {list(Xf.columns)[:12]}\nY cols: {list(Yf.columns)[:12]}")

    if key == "id":
        Xf[key] = pd.to_numeric(Xf[key], errors="coerce").astype("Int64")
        Yf[key] = pd.to_numeric(Yf[key], errors="coerce").astype("Int64")
    else:
        Xf[key] = Xf[key].astype(str).str.strip()
        Yf[key] = Yf[key].astype(str).str.strip()

    if required_obj_cols:
        miss = [c for c in required_obj_cols if c not in Yf.columns]
        if miss:
            raise ValueError(f"Objectives missing: {miss}\nAvailable: {list(Yf.columns)[:25]}")

    cols_to_merge = [key] + (required_obj_cols or [])
    for obj_col in (required_obj_cols or []):
        if obj_col.endswith("_obj"):
            sc = obj_col.replace("_obj", "_status")
            if sc in Yf.columns:
                cols_to_merge.append(sc)

    df = pd.merge(Xf, Yf[cols_to_merge], on=key, how="inner")
    df["id_key"] = df[key].astype(str).str.strip()
    return df, key

def load_merge_one(features_path, objectives_path, prefer_key="id_global", required_obj_cols=None):
    Xf = read_any(features_path)
    Yf = normalize_objective_columns(read_any(objectives_path))
    df, used_key = merge_features_objectives(Xf, Yf, prefer=prefer_key, required_obj_cols=required_obj_cols)
    df["_source_feat"] = Path(features_path).name
    df["_source_obj"]  = Path(objectives_path).name
    df["_merge_key"]   = used_key
    print(f"[{_ts()}] [merge] {Path(features_path).name} + {Path(objectives_path).name} -> rows={len(df)}", flush=True)
    if len(df) == 0:
        print(f"[{_ts()}] [merge][WARN] 0 rows!", flush=True)
    return df

def concat_pairs(feature_paths, objective_paths, prefer_key="id_global", required_obj_cols=None):
    if len(feature_paths) != len(objective_paths):
        raise ValueError("Counts must match.")
    frames = [load_merge_one(fx, oy, prefer_key=prefer_key, required_obj_cols=required_obj_cols)
              for fx, oy in zip(feature_paths, objective_paths)]
    df = pd.concat(frames, ignore_index=True)
    if "id_key" not in df.columns:
        raise ValueError("'id_key' not created.")
    return df

# ============================================================
# COLUMN DROPS / FILTERS
# ============================================================
def drop_unused_relaxation_cols(df, used_relaxations):
    df = df.copy()
    unused = [r for r in ALL_RELAXATIONS if r not in used_relaxations]
    to_drop = sorted({c for r in unused for c in df.columns if c.startswith(f"{r}_")})
    if to_drop:
        df = df.drop(columns=to_drop, errors="ignore")
        print(f"[{_ts()}] [filter] Dropped {len(to_drop)} cols for: {unused}", flush=True)
    return df

def drop_metadata_cols(df):
    df = df.copy()
    to_drop = sorted({
        c for c in df.columns
        if (
            str(c).endswith(("_runtime", "_t", "_status", "_relaxation"))
            or str(c).startswith(("bt_", "bigM_", "san_"))
            or str(c) in ("n_inf_after_bt", "Iteration")
        )
    })
    if to_drop:
        df = df.drop(columns=to_drop, errors="ignore")
    return df

def filter_by_status(df, relaxations, exclude_statuses):
    """
    Remove only truly excluded statuses.
    Unbounded rows are NOT removed here.
    """
    df = df.copy()
    n_before = len(df)
    bad = pd.Series(False, index=df.index)

    for r in relaxations:
        sc = f"{r}_status"
        oc = f"{r}_obj"

        if sc in df.columns:
            bad |= df[sc].map(_norm_status).isin(exclude_statuses)

        if oc in df.columns:
            status = df[sc].map(_norm_status) if sc in df.columns else pd.Series("", index=df.index)
            bad |= (status == "") & df[oc].isna()

    df = df[~bad].copy()
    removed = n_before - len(df)
    if removed:
        print(f"[{_ts()}] [filter] Removed {removed} rows ({len(df)} remain)", flush=True)
    return df

# ============================================================
# TARGET CREATION
# ============================================================
def _compute_delta_fixed(a: np.ndarray, b: np.ndarray, epsilon_gap: float) -> np.ndarray:
    denom = np.abs(a) + np.abs(b) + float(epsilon_gap)
    return np.where(np.isfinite(a - b), (a - b) / denom, 0.0)

def _transform_delta(delta, transform):
    t = (transform or "none").strip().lower()
    if t == "none":
        return delta
    if t == "clip":
        return np.clip(delta, -1.0, 1.0)
    return np.tanh(delta)

def make_pairwise_target(
    df,
    relax_a,
    relax_b,
    epsilon_gap=1e-3,
    epsilon_method="constant",
    delta_transform="none",
    precomputed_floor=None,
):
    """
    Keep unbounded rows with explicit targets:
        relax_b unbounded + relax_a finite -> +1.0
        relax_a unbounded + relax_b finite -> -1.0
        both finite                        -> fixed delta
    Only truly unusable rows stay NaN and are removed.
    finite_mask == 1 only for finite/finite rows.
    """
    if epsilon_method != "constant":
        raise ValueError(
            f"epsilon_method='{epsilon_method}' breaks alignment. Use 'constant'.")

    df = df.copy()

    col_a = f"{relax_a}_obj"
    col_b = f"{relax_b}_obj"
    st_a  = f"{relax_a}_status"
    st_b  = f"{relax_b}_status"

    df[col_a] = pd.to_numeric(df.get(col_a), errors="coerce")
    df[col_b] = pd.to_numeric(df.get(col_b), errors="coerce")

    a = df[col_a].astype(float).values
    b = df[col_b].astype(float).values

    a_unb = (_norm_status_series(df[st_a]).isin(UNBOUNDED_STATUSES).values | np.isneginf(a)) \
            if st_a in df.columns else np.isneginf(a)
    b_unb = (_norm_status_series(df[st_b]).isin(UNBOUNDED_STATUSES).values | np.isneginf(b)) \
            if st_b in df.columns else np.isneginf(b)

    a_fin = np.isfinite(a) & ~a_unb
    b_fin = np.isfinite(b) & ~b_unb

    y          = np.full(len(df), np.nan)
    case       = np.full(len(df), "excluded", dtype=object)
    finite_mask = np.zeros(len(df), dtype=int)

    # unbounded explicit targets
    mask_b_unb = b_unb & a_fin
    mask_a_unb = a_unb & b_fin

    y[mask_b_unb]    = _transform_delta(+1.0, delta_transform)
    case[mask_b_unb] = f"{relax_b}_unbounded"

    y[mask_a_unb]    = _transform_delta(-1.0, delta_transform)
    case[mask_a_unb] = f"{relax_a}_unbounded"

    # finite / finite
    finite = a_fin & b_fin
    if np.any(finite):
        d = _compute_delta_fixed(a[finite], b[finite], epsilon_gap)
        y[finite] = _transform_delta(d, delta_transform)
        case[finite] = "finite"
        finite_mask[finite] = 1

    df["y"] = y
    df["delta"] = y
    df["target_case"] = case
    df["finite_mask"] = finite_mask

    before = len(df)
    df = df[df["y"].notna()].copy()
    removed = before - len(df)

    print(
        f"[{_ts()}] [target] pairwise method=fixed_additive gap={epsilon_gap} "
        f"transform={delta_transform} | kept {len(df)}/{before} removed={removed}",
        flush=True,
    )
    print(f"[{_ts()}] [target] case counts: {df['target_case'].value_counts().to_dict()}", flush=True)

    return df, None

def make_multi_targets(
    df,
    relaxations,
    baseline,
    epsilon_gap=1e-3,
    epsilon_method="constant",
    delta_transform="none",
    precomputed_floors=None,
):
    """
    Multi-target version.
    For each target y_r comparing baseline vs r:
        r unbounded + baseline finite -> +1
        baseline unbounded + r finite -> -1
        both finite                   -> fixed delta
    We also create finite-mask columns for finite-only evaluation.
    """
    if epsilon_method != "constant":
        raise ValueError(
            f"epsilon_method='{epsilon_method}' breaks alignment. Use 'constant'.")

    df = df.copy()

    base_obj  = f"{baseline}_obj"
    base_stat = f"{baseline}_status"
    df[base_obj] = pd.to_numeric(df.get(base_obj), errors="coerce")
    base = df[base_obj].astype(float).values

    base_unb = (_norm_status_series(df[base_stat]).isin(UNBOUNDED_STATUSES).values | np.isneginf(base)) \
               if base_stat in df.columns else np.isneginf(base)
    base_fin = np.isfinite(base) & ~base_unb

    target_cols = []
    finite_mask_cols = []

    for r in relaxations:
        tcol = f"y_{r}"
        mcol = f"finite_mask_{r}"
        target_cols.append(tcol)
        finite_mask_cols.append(mcol)

        if r == baseline:
            df[tcol] = 0.0
            df[mcol] = 0
            continue

        r_obj  = f"{r}_obj"
        r_stat = f"{r}_status"

        if r_obj not in df.columns:
            df[tcol] = np.nan
            df[mcol] = 0
            continue

        df[r_obj] = pd.to_numeric(df.get(r_obj), errors="coerce")
        rv = df[r_obj].astype(float).values

        r_unb = (_norm_status_series(df[r_stat]).isin(UNBOUNDED_STATUSES).values | np.isneginf(rv)) \
                if r_stat in df.columns else np.isneginf(rv)
        r_fin = np.isfinite(rv) & ~r_unb

        y = np.full(len(df), np.nan)
        m = np.zeros(len(df), dtype=int)

        mask_r_unb = r_unb & base_fin
        mask_b_unb = base_unb & r_fin

        y[mask_r_unb] = _transform_delta(+1.0, delta_transform)
        y[mask_b_unb] = _transform_delta(-1.0, delta_transform)

        finite = base_fin & r_fin
        if np.any(finite):
            d = _compute_delta_fixed(base[finite], rv[finite], epsilon_gap)
            y[finite] = _transform_delta(d, delta_transform)
            m[finite] = 1

        df[tcol] = y
        df[mcol] = m

    before = len(df)
    df = df[df[target_cols].notna().any(axis=1)].copy()
    removed = before - len(df)

    print(
        f"[{_ts()}] [target] multi method=fixed_additive gap={epsilon_gap} "
        f"transform={delta_transform} | kept {len(df)}/{before} removed={removed}",
        flush=True,
    )
    return df, target_cols, {"finite_mask_cols": finite_mask_cols}

# ============================================================
# METRICS
# ============================================================
def r_accuracy(y_true, y_pred, eps=TAU_SIGN):
    y = np.asarray(y_true, dtype=float)
    yhat = np.asarray(y_pred, dtype=float)
    m = np.isfinite(y) & np.isfinite(yhat)
    y = y[m]
    yhat = yhat[m]
    true_lp = y > -eps
    pred_lp = yhat > -eps
    return float(np.mean(true_lp == pred_lp)) if len(y) > 0 else float("nan")

def safe_pearson_r(y_true, y_pred):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    m = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[m]
    yp = yp[m]
    if len(yt) == 0 or np.std(yt) <= 0 or np.std(yp) <= 0:
        return float("nan")
    r = np.corrcoef(yt, yp)[0, 1]
    return float(r) if np.isfinite(r) else float("nan")

def sign_accuracy_tau(y_true, y_pred, tau=TAU_SIGN):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp) & (np.abs(yt) >= tau)
    if mask.sum() == 0:
        return float("nan"), 0.0
    true_lp = yt[mask] > -tau
    pred_lp = yp[mask] > -tau
    return float(np.mean(true_lp == pred_lp)), float(mask.mean())

def save_sign_confusion_matrix(run_dir, name, run_tag, y_true, y_pred, tau, relax_a, relax_b):
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[mask]
    yp = yp[mask]

    true_dec = (yt > -tau).astype(int)
    pred_dec = (yp > -tau).astype(int)

    cm = confusion_matrix(true_dec, pred_dec, labels=[0, 1])
    acc = float(np.mean(true_dec == pred_dec)) if len(true_dec) > 0 else float("nan")

    pd.DataFrame(
        cm,
        index=[f"true_{relax_b}(0)", f"true_{relax_a}(1)"],
        columns=[f"pred_{relax_b}(0)", f"pred_{relax_a}(1)"],
    ).to_csv(run_dir / f"sign_cm_{name}_{run_tag}.csv", index=True)

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=[relax_b, relax_a]).plot(ax=ax)
    ax.set_title(f"LP-biased sign @ tau={tau:.3f}  acc={acc:.3f}")
    plt.tight_layout()
    plt.savefig(run_dir / f"sign_cm_{name}_{run_tag}.png", dpi=200)
    plt.close()

# ============================================================
# PREPROCESSING
# ============================================================
def build_preprocessor(X):
    cat_cols = [
        c for c in X.columns
        if pd.api.types.is_object_dtype(X[c]) or isinstance(X[c].dtype, CategoricalDtype)
    ]
    num_cols = [c for c in X.columns if c not in cat_cols]

    transformers = []
    if cat_cols:
        try:
            ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        except Exception:
            ohe = OneHotEncoder(sparse=False, handle_unknown="ignore")
        transformers.append((
            "cat",
            Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("ohe", ohe),
            ]),
            cat_cols,
        ))
    transformers.append((
        "num",
        Pipeline([
            ("imp", SimpleImputer(strategy="median")),
        ]),
        num_cols,
    ))
    return ColumnTransformer(transformers, remainder="drop"), cat_cols, num_cols

def align_test_to_train_cols(X_train, X_test):
    X_test = X_test.copy()
    for c in [c for c in X_train.columns if c not in X_test.columns]:
        X_test[c] = np.nan
    return X_test[X_train.columns]

# ============================================================
# SAVE OUTPUTS
# ============================================================
def save_pairwise_outputs(run_dir, name, run_tag, ids, y_true, y_pred, finite_mask, target_case, meta=None):
    df = pd.DataFrame({
        KEY_COL: ids.astype(str),
        "y_true": y_true,
        "y_pred": y_pred,
        "finite_mask": finite_mask,
        "target_case": target_case,
    })
    if meta:
        for c, v in meta.items():
            df[c] = v
    df["err"] = df["y_pred"] - df["y_true"]
    df.to_csv(run_dir / f"preds_{name}_{run_tag}.csv", index=False)

    plt.figure(figsize=(5.4, 5.4))
    plt.scatter(y_true, y_pred, s=8, alpha=0.5)
    lo = min(np.nanmin(y_true), np.nanmin(y_pred))
    hi = max(np.nanmax(y_true), np.nanmax(y_pred))
    plt.plot([lo, hi], [lo, hi], "r--", label="Perfect")
    plt.axhline(0, ls="--", color="gray", alpha=0.5)
    plt.axvline(0, ls="--", color="gray", alpha=0.5)
    plt.axhline(-TAU_SIGN, ls=":", color="blue", alpha=0.4, label=f"-tau={-TAU_SIGN}")
    plt.axvline(-TAU_SIGN, ls=":", color="blue", alpha=0.4)
    plt.xlabel("True target")
    plt.ylabel("Predicted target")
    plt.title(f"Actual vs Predicted ({name})")
    plt.legend()
    plt.tight_layout()
    plt.savefig(run_dir / f"scatter_{name}_{run_tag}.png", dpi=200)
    plt.close()

def save_multi_outputs(run_dir, name, run_tag, ids, y_true, y_pred, target_names, meta=None):
    df = pd.DataFrame({KEY_COL: ids.astype(str)})
    for i, t in enumerate(target_names):
        df[f"{t}_true"] = y_true[:, i]
        df[f"{t}_pred"] = y_pred[:, i]
        df[f"{t}_err"] = y_pred[:, i] - y_true[:, i]
    if meta:
        for c, v in meta.items():
            df[c] = v
    df.to_csv(run_dir / f"preds_{name}_{run_tag}.csv", index=False)

    n = len(target_names)
    fig, axes = plt.subplots(1, min(n, 4), figsize=(5 * min(n, 4), 5))
    if n == 1:
        axes = [axes]
    for i, (ax, t) in enumerate(zip(axes.flat if n > 1 else axes, target_names[:4])):
        ax.scatter(y_true[:, i], y_pred[:, i], s=8, alpha=0.5)
        lo = min(np.nanmin(y_true[:, i]), np.nanmin(y_pred[:, i]))
        hi = max(np.nanmax(y_true[:, i]), np.nanmax(y_pred[:, i]))
        ax.plot([lo, hi], [lo, hi], "r--")
        ax.axhline(0, ls="--", color="gray", alpha=0.5)
        ax.axvline(0, ls="--", color="gray", alpha=0.5)
        ax.axhline(-TAU_SIGN, ls=":", color="blue", alpha=0.4)
        ax.axvline(-TAU_SIGN, ls=":", color="blue", alpha=0.4)
        ax.set_xlabel("True")
        ax.set_ylabel("Predicted")
        ax.set_title(t)
    plt.tight_layout()
    plt.savefig(run_dir / f"scatter_{name}_{run_tag}.png", dpi=200)
    plt.close()

# ============================================================
# SAMPLE WEIGHTS
# ============================================================
def _make_sample_weights_pairwise(y, alpha):
    a = min(max(float(alpha), 0.0), 1.0)
    strength = np.abs(np.asarray(y, dtype=float))
    strength = np.where(np.isfinite(strength), strength, 0.0)
    return (1.0 - a) + a * strength

def _make_sample_weights_multi(Y, alpha):
    a = min(max(float(alpha), 0.0), 1.0)
    strength = np.max(np.abs(np.asarray(Y, dtype=float)), axis=1)
    strength = np.where(np.isfinite(strength), strength, 0.0)
    return (1.0 - a) + a * strength

# ============================================================
# TRAIN + EVAL
# ============================================================
_ALWAYS_DROP_PAIRWISE = [
    "delta",
    "y",
    "_source_feat",
    "_source_obj",
    "_merge_key",
    "target_case",
    "finite_mask",
]
_ID_LIKE = ["id", "id_global", "instance_id", "instanceid", "instance_name"]

def train_eval_regression(
    train_df,
    test_df,
    key,
    run_dir,
    run_tag,
    mode,
    pairwise_relaxations,
    multi_relaxations,
    multi_baseline,
    epsilon_gap=1e-3,
    epsilon_method="constant",
    delta_transform="none",
    selected_features=None,
):
    print(f"[{_ts()}] ═══════════════════════════════════════════════════", flush=True)
    print(f"[{_ts()}] EPSILON_METHOD={epsilon_method} (fixed additive)", flush=True)
    print(f"[{_ts()}] GAP={epsilon_gap} | TRANSFORM={delta_transform}", flush=True)
    print(f"[{_ts()}] TAU_SIGN={TAU_SIGN} | decision_rule=LP-biased (delta > -tau)", flush=True)
    print(f"[{_ts()}] SAMPLE_WEIGHTS={USE_SAMPLE_WEIGHTS} | ALPHA={WEIGHT_ALPHA}", flush=True)
    print(f"[{_ts()}] KEEP unbounded rows with explicit targets; ignore them in MSE", flush=True)

    if epsilon_method != "constant":
        raise ValueError(f"epsilon_method must be 'constant'. Got '{epsilon_method}'")

    if mode == "pairwise":
        used_relaxations = list(pairwise_relaxations)
        print(f"[{_ts()}] PAIRWISE: {pairwise_relaxations[0]} vs {pairwise_relaxations[1]}", flush=True)
    else:
        used_relaxations = multi_relaxations
        print(f"[{_ts()}] MULTI: {len(multi_relaxations)} targets vs baseline={multi_baseline}", flush=True)

    n_feat_str = f"{len(selected_features)} selected" if selected_features else "ALL"
    print(f"[{_ts()}] FEATURES: {n_feat_str}", flush=True)
    print(f"[{_ts()}] ═══════════════════════════════════════════════════", flush=True)

    # Step 1: remove only excluded statuses
    train_df = filter_by_status(train_df, used_relaxations, EXCLUDE_STATUSES)
    if test_df is not None:
        test_df = filter_by_status(test_df, used_relaxations, EXCLUDE_STATUSES)

    # Step 2: build targets BEFORE X/y extraction and BEFORE splitting
    target_cols = None
    extra_target_info = {}

    if mode == "pairwise":
        train_df, _ = make_pairwise_target(
            train_df,
            pairwise_relaxations[0],
            pairwise_relaxations[1],
            epsilon_gap=epsilon_gap,
            epsilon_method=epsilon_method,
            delta_transform=delta_transform,
        )
        if test_df is not None:
            test_df, _ = make_pairwise_target(
                test_df,
                pairwise_relaxations[0],
                pairwise_relaxations[1],
                epsilon_gap=epsilon_gap,
                epsilon_method=epsilon_method,
                delta_transform=delta_transform,
            )
    else:
        train_df, target_cols, extra_target_info = make_multi_targets(
            train_df,
            multi_relaxations,
            multi_baseline,
            epsilon_gap=epsilon_gap,
            epsilon_method=epsilon_method,
            delta_transform=delta_transform,
        )
        if test_df is not None:
            test_df, _, _ = make_multi_targets(
                test_df,
                multi_relaxations,
                multi_baseline,
                epsilon_gap=epsilon_gap,
                epsilon_method=epsilon_method,
                delta_transform=delta_transform,
            )

    train_df = drop_unused_relaxation_cols(train_df, used_relaxations)
    train_df = drop_metadata_cols(train_df)
    if test_df is not None:
        test_df = drop_unused_relaxation_cols(test_df, used_relaxations)
        test_df = drop_metadata_cols(test_df)

    if len(train_df) == 0:
        raise ValueError("train_df is empty after filtering/target creation")

    CARRY_COLS = [
        c for c in ("instance_name", "n", "m", "id", "id_global", "_merge_key", "target_case", "finite_mask")
        if c in train_df.columns or (test_df is not None and c in test_df.columns)
    ]
    obj_cols = [f"{r}_obj" for r in used_relaxations]

    if mode == "pairwise":
        drop_cols = list({key} | set(obj_cols) | set(_ALWAYS_DROP_PAIRWISE) | set(_ID_LIKE))
    else:
        finite_mask_cols = extra_target_info.get("finite_mask_cols", [])
        drop_cols = list(
            {key}
            | set(obj_cols)
            | set(target_cols)
            | set(finite_mask_cols)
            | {"_source_feat", "_source_obj", "_merge_key"}
            | set(_ID_LIKE)
        )

    drop_cols = sorted(set(c for c in drop_cols if c in train_df.columns))

    Xtr = train_df.drop(columns=drop_cols, errors="ignore")
    Xtr = filter_to_selected_features(Xtr, selected_features)
    Xtr = Xtr.replace([np.inf, -np.inf], np.nan)
    ids_tr = train_df.loc[Xtr.index, key].astype(str)

    if mode == "pairwise":
        ytr = train_df["y"].astype(float).values
        finite_mask_tr_all = train_df["finite_mask"].astype(int).values
        case_tr_all = train_df["target_case"].astype(str).values
    else:
        ytr = train_df[target_cols].astype(float).values
        finite_mask_cols = [c for c in train_df.columns if c.startswith("finite_mask_")]
        finite_mask_tr_all = train_df[finite_mask_cols].astype(int).values
        case_tr_all = None

    print(f"[{_ts()}] [features] Train shape: {Xtr.shape}", flush=True)
    if Xtr.shape[1] == 0:
        raise ValueError("No features remaining")

    pre, _, _ = build_preprocessor(Xtr)

    def _make_models(pre):
        m = {
            "gbr": Pipeline([
                ("pre", pre),
                ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
            ]),
            "rf": Pipeline([
                ("pre", pre),
                ("model", RandomForestRegressor(n_estimators=400, n_jobs=-1, random_state=RANDOM_STATE)),
            ]),
        }
        if XGBOOST_AVAILABLE:
            m["xgb"] = Pipeline([
                ("pre", pre),
                ("model", XGBRegressor(
                    n_estimators=400,
                    learning_rate=0.08,
                    max_depth=6,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_alpha=0.1,
                    reg_lambda=1.0,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )),
            ])
            print(f"[{_ts()}] [models] XGBoost enabled", flush=True)
        else:
            print(f"[{_ts()}] [models] XGBoost not available", flush=True)
        return m

    if mode == "pairwise":
        models = _make_models(pre)
    else:
        base_gbr = GradientBoostingRegressor(random_state=RANDOM_STATE)
        base_rf  = RandomForestRegressor(n_estimators=400, n_jobs=-1, random_state=RANDOM_STATE)
        models = {
            "gbr": Pipeline([("pre", pre), ("model", MultiOutputRegressor(base_gbr))]),
            "rf":  Pipeline([("pre", pre), ("model", MultiOutputRegressor(base_rf))]),
        }
        if XGBOOST_AVAILABLE:
            base_xgb = XGBRegressor(
                n_estimators=400,
                learning_rate=0.08,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
            models["xgb"] = Pipeline([("pre", pre), ("model", MultiOutputRegressor(base_xgb))])

    results = {}

    # ========================================================
    # HOLDOUT
    # ========================================================
    if test_df is None:
        tr_idx, te_idx = make_deterministic_split(ids_tr, test_frac=TEST_SIZE)

        X_tr = Xtr.iloc[tr_idx]
        X_te = Xtr.iloc[te_idx]
        ids_te = ids_tr.iloc[te_idx]

        if mode == "pairwise":
            y_tr = ytr[tr_idx]
            y_te = ytr[te_idx]
            finite_mask_te = finite_mask_tr_all[te_idx]
            case_te = case_tr_all[te_idx]
        else:
            y_tr = ytr[tr_idx]
            y_te = ytr[te_idx]
            finite_mask_te = finite_mask_tr_all[te_idx, :]

        print(f"[{_ts()}] [split] Holdout (hash-based): train={len(y_tr)}, test={len(y_te)}", flush=True)

        w_tr = None
        if USE_SAMPLE_WEIGHTS:
            w_tr = (_make_sample_weights_pairwise(y_tr, WEIGHT_ALPHA)
                    if mode == "pairwise"
                    else _make_sample_weights_multi(y_tr, WEIGHT_ALPHA))

        for name, pipe in models.items():
            print(f"[{_ts()}] [train] {name} ...", flush=True)
            pipe.fit(X_tr, y_tr, **({"model__sample_weight": w_tr} if w_tr is not None else {}))
            y_hat = pipe.predict(X_te)

            if mode == "pairwise":
                finite_eval = np.asarray(finite_mask_te, dtype=int) == 1

                y_te_f = y_te[finite_eval]
                y_hat_f = y_hat[finite_eval]

                mse_f = mean_squared_error(y_te_f, y_hat_f) if finite_eval.sum() > 0 else float("nan")
                mae_f = mean_absolute_error(y_te_f, y_hat_f) if finite_eval.sum() > 0 else float("nan")
                r_f   = safe_pearson_r(y_te_f, y_hat_f)

                racc = r_accuracy(y_te, y_hat)
                sacc, scov = sign_accuracy_tau(y_te, y_hat, tau=TAU_SIGN)

                results[name] = dict(
                    model=name,
                    mode=mode,
                    n_features=Xtr.shape[1],
                    n_train=len(y_tr),
                    n_test=len(y_te),
                    mse=mse_f,
                    mae=mae_f,
                    r=r_f,
                    r_accuracy=racc,
                    sign_acc_tau=sacc,
                    sign_cov=scov,
                    tau_sign=TAU_SIGN,
                    decision_rule="lp_biased",
                    eps_gap=epsilon_gap,
                    eps_method=epsilon_method,
                    delta_transform=delta_transform,
                    sample_weights=USE_SAMPLE_WEIGHTS,
                    weight_alpha=WEIGHT_ALPHA,
                    n_finite=int(np.sum(case_te == "finite")),
                    n_sdp_unbounded=int(np.sum(case_te == f"{pairwise_relaxations[1]}_unbounded")),
                    n_lp_unbounded=int(np.sum(case_te == f"{pairwise_relaxations[0]}_unbounded")),
                )

                print(
                    f"[{_ts()}] [eval] {name}: mse_finite={mse_f:.4f} mae_finite={mae_f:.4f} "
                    f"r_finite={r_f:.3f} r_acc={racc:.3f} sign@tau={sacc:.3f} (cov={scov:.2f})",
                    flush=True,
                )

                save_pairwise_outputs(run_dir, name, run_tag, ids_te, y_te, y_hat, finite_mask_te, case_te)
                save_sign_confusion_matrix(
                    run_dir, name, run_tag, y_te, y_hat, TAU_SIGN,
                    pairwise_relaxations[0], pairwise_relaxations[1]
                )

            else:
                finite_eval = np.asarray(finite_mask_te, dtype=int) == 1
                y_te_f = y_te[finite_eval]
                y_hat_f = y_hat[finite_eval]

                if y_te_f.size > 0:
                    mse_o = mean_squared_error(y_te_f, y_hat_f)
                    mae_o = mean_absolute_error(y_te_f, y_hat_f)
                else:
                    mse_o = float("nan")
                    mae_o = float("nan")

                results[name] = dict(
                    model=name,
                    mode=mode,
                    n_features=Xtr.shape[1],
                    n_targets=y_tr.shape[1],
                    n_train=len(y_tr),
                    n_test=len(y_te),
                    mse_overall=mse_o,
                    mae_overall=mae_o,
                    eps_gap=epsilon_gap,
                    eps_method=epsilon_method,
                    delta_transform=delta_transform,
                    sample_weights=USE_SAMPLE_WEIGHTS,
                    weight_alpha=WEIGHT_ALPHA,
                )

                print(f"[{_ts()}] [eval] {name}: mse_finite={mse_o:.4f} mae_finite={mae_o:.4f}", flush=True)
                save_multi_outputs(run_dir, name, run_tag, ids_te, y_te, y_hat, target_cols)

    # ========================================================
    # EXTERNAL TEST
    # ========================================================
    else:
        if len(test_df) == 0:
            raise ValueError("test_df is empty after filtering/target creation")

        drop_cols_te = [c for c in drop_cols if c in test_df.columns]
        Xte = test_df.drop(columns=drop_cols_te, errors="ignore")
        Xte = filter_to_selected_features(Xte, selected_features)
        Xte = Xte.replace([np.inf, -np.inf], np.nan)
        Xte = align_test_to_train_cols(Xtr, Xte)
        ids_te = test_df.loc[Xte.index, key].astype(str)

        if mode == "pairwise":
            yte = test_df["y"].astype(float).values
            finite_mask_te = test_df["finite_mask"].astype(int).values
            case_te = test_df["target_case"].astype(str).values
        else:
            yte = test_df[target_cols].astype(float).values
            finite_mask_cols = [c for c in test_df.columns if c.startswith("finite_mask_")]
            finite_mask_te = test_df[finite_mask_cols].astype(int).values

        meta_te = {c: test_df.loc[Xte.index, c].values for c in CARRY_COLS if c in test_df.columns}
        print(f"[{_ts()}] [test] External test shape: {Xte.shape}", flush=True)

        w_full = None
        if USE_SAMPLE_WEIGHTS:
            w_full = (_make_sample_weights_pairwise(ytr, WEIGHT_ALPHA)
                      if mode == "pairwise"
                      else _make_sample_weights_multi(ytr, WEIGHT_ALPHA))

        for name, pipe in models.items():
            print(f"[{_ts()}] [train] {name} on full train ...", flush=True)
            pipe.fit(Xtr, ytr, **({"model__sample_weight": w_full} if w_full is not None else {}))
            y_hat = pipe.predict(Xte)

            if mode == "pairwise":
                finite_eval = np.asarray(finite_mask_te, dtype=int) == 1

                yte_f = yte[finite_eval]
                yhat_f = y_hat[finite_eval]

                mse_f = mean_squared_error(yte_f, yhat_f) if finite_eval.sum() > 0 else float("nan")
                mae_f = mean_absolute_error(yte_f, yhat_f) if finite_eval.sum() > 0 else float("nan")
                r_f   = safe_pearson_r(yte_f, yhat_f)

                racc = r_accuracy(yte, y_hat)
                sacc, scov = sign_accuracy_tau(yte, y_hat, tau=TAU_SIGN)

                results[name] = dict(
                    model=name,
                    mode=mode,
                    n_features=Xtr.shape[1],
                    n_train=len(ytr),
                    n_test=len(yte),
                    mse=mse_f,
                    mae=mae_f,
                    r=r_f,
                    r_accuracy=racc,
                    sign_acc_tau=sacc,
                    sign_cov=scov,
                    tau_sign=TAU_SIGN,
                    decision_rule="lp_biased",
                    note="external test",
                    eps_gap=epsilon_gap,
                    eps_method=epsilon_method,
                    delta_transform=delta_transform,
                    sample_weights=USE_SAMPLE_WEIGHTS,
                    weight_alpha=WEIGHT_ALPHA,
                    n_finite=int(np.sum(case_te == "finite")),
                    n_sdp_unbounded=int(np.sum(case_te == f"{pairwise_relaxations[1]}_unbounded")),
                    n_lp_unbounded=int(np.sum(case_te == f"{pairwise_relaxations[0]}_unbounded")),
                )

                print(
                    f"[{_ts()}] [eval] {name}: mse_finite={mse_f:.4f} mae_finite={mae_f:.4f} "
                    f"r_finite={r_f:.3f} r_acc={racc:.3f} sign@tau={sacc:.3f} (cov={scov:.2f})",
                    flush=True,
                )

                save_pairwise_outputs(run_dir, name, run_tag, ids_te, yte, y_hat, finite_mask_te, case_te, meta_te)
                save_sign_confusion_matrix(
                    run_dir, name, run_tag, yte, y_hat, TAU_SIGN,
                    pairwise_relaxations[0], pairwise_relaxations[1]
                )

            else:
                finite_eval = np.asarray(finite_mask_te, dtype=int) == 1
                yte_f = yte[finite_eval]
                yhat_f = y_hat[finite_eval]

                if yte_f.size > 0:
                    mse_o = mean_squared_error(yte_f, yhat_f)
                    mae_o = mean_absolute_error(yte_f, yhat_f)
                else:
                    mse_o = float("nan")
                    mae_o = float("nan")

                results[name] = dict(
                    model=name,
                    mode=mode,
                    n_features=Xtr.shape[1],
                    n_targets=ytr.shape[1],
                    n_train=len(ytr),
                    n_test=len(yte),
                    mse_overall=mse_o,
                    mae_overall=mae_o,
                    note="external test",
                    eps_gap=epsilon_gap,
                    eps_method=epsilon_method,
                    delta_transform=delta_transform,
                    sample_weights=USE_SAMPLE_WEIGHTS,
                    weight_alpha=WEIGHT_ALPHA,
                )

                print(f"[{_ts()}] [eval] {name}: mse_finite={mse_o:.4f} mae_finite={mae_o:.4f}", flush=True)
                save_multi_outputs(run_dir, name, run_tag, ids_te, yte, y_hat, target_cols, meta_te)

    pd.DataFrame.from_dict(results, orient="index").reset_index(drop=False) \
        .rename(columns={"index": "model"}) \
        .to_csv(run_dir / f"results_reg_{run_tag}.csv", index=False)

    return results

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    print(f"[{_ts()}] ═══════════════════════════════════════════════════", flush=True)
    print(f"[{_ts()}] REGRESSION KEEP-UNBOUNDED  GAP=fixed_additive", flush=True)
    print(f"[{_ts()}] TAU_SIGN={TAU_SIGN}  EPS_GAP={EPSILON_GAP}", flush=True)
    print(f"[{_ts()}] decision_rule=LP-biased (delta > -TAU_SIGN)", flush=True)
    print(f"[{_ts()}] unbounded rows kept with explicit targets; ignored in finite MSE", flush=True)
    print(f"[{_ts()}] ═══════════════════════════════════════════════════", flush=True)

    SELECTED_FEATURES = load_features_from_txt(FEATURE_FILE)

    if MODE not in ("pairwise", "multi"):
        raise ValueError(f"MODE must be 'pairwise' or 'multi', got: {MODE}")

    if MODE == "pairwise":
        active_relaxations = list(PAIRWISE_RELAXATIONS)
        print(f"[{_ts()}] MODE=pairwise | relaxations={PAIRWISE_RELAXATIONS}", flush=True)
    else:
        active_relaxations = MULTI_RELAXATIONS
        print(f"[{_ts()}] MODE=multi | relaxations={MULTI_RELAXATIONS} | baseline={MULTI_BASELINE}", flush=True)

    print(f"[{_ts()}] EPSILON_GAP={EPSILON_GAP} | EPSILON_METHOD={EPSILON_METHOD}", flush=True)
    print(f"[{_ts()}] DELTA_TRANSFORM={DELTA_TRANSFORM} | TAU_SIGN={TAU_SIGN}", flush=True)
    print(f"[{_ts()}] USE_SAMPLE_WEIGHTS={USE_SAMPLE_WEIGHTS} | WEIGHT_ALPHA={WEIGHT_ALPHA}", flush=True)
    print(f"[{_ts()}] ═══════════════════════════════════════════════════", flush=True)

    required_obj_cols = [f"{r}_obj" for r in active_relaxations]
    train_tag = run_tag_from_lists(TRAIN_FEATURES)
    test_tag  = run_tag_from_lists(TEST_FEATURES) if not USE_HOLDOUT_FROM_TRAIN else "holdout"

    if MODE == "pairwise":
        RUN_TAG = (
            f"{SETUP}__pairwise__{PAIRWISE_RELAXATIONS[0]}v{PAIRWISE_RELAXATIONS[1]}"
            f"__train[{train_tag}]__test[{test_tag}]"
        )
    else:
        RUN_TAG = (
            f"{SETUP}__multi__{'_'.join(MULTI_RELAXATIONS)}"
            f"__baseline_{MULTI_BASELINE}"
            f"__train[{train_tag}]__test[{test_tag}]"
        )

    RUN_DIR = OUTPUT_ROOT / RUN_TAG
    RUN_DIR.mkdir(exist_ok=True, parents=True)

    print(f"[{_ts()}] Loading training data ...", flush=True)
    train_df = concat_pairs(
        TRAIN_FEATURES,
        TRAIN_OBJECTS,
        prefer_key="id_global",
        required_obj_cols=required_obj_cols,
    )

    test_df = None
    if not USE_HOLDOUT_FROM_TRAIN:
        print(f"[{_ts()}] Loading test data ...", flush=True)
        test_df = concat_pairs(
            TEST_FEATURES,
            TEST_OBJECTS,
            prefer_key="id_global",
            required_obj_cols=required_obj_cols,
        )

    print(f"[{_ts()}] Train rows: {len(train_df)} | Test rows: {len(test_df) if test_df is not None else 'holdout'}", flush=True)
    print(f"[{_ts()}] Run tag : {RUN_TAG}", flush=True)
    print(f"[{_ts()}] Output  : {RUN_DIR.resolve()}", flush=True)

    results = train_eval_regression(
        train_df,
        test_df,
        KEY_COL,
        RUN_DIR,
        RUN_TAG,
        MODE,
        PAIRWISE_RELAXATIONS if MODE == "pairwise" else None,
        MULTI_RELAXATIONS if MODE == "multi" else None,
        MULTI_BASELINE if MODE == "multi" else None,
        EPSILON_GAP,
        EPSILON_METHOD,
        DELTA_TRANSFORM,
        SELECTED_FEATURES,
    )

    print(f"[{_ts()}] ═══════════════════════════════════════════════════", flush=True)
    print(f"[{_ts()}] FINAL RESULTS:", flush=True)
    for model, m in results.items():
        if MODE == "pairwise":
            print(
                f"[{_ts()}] {model}: mse_finite={m['mse']:.4f} mae_finite={m['mae']:.4f} "
                f"r_finite={m['r']:.3f} r_acc={m['r_accuracy']:.3f} "
                f"sign@tau={m.get('sign_acc_tau', float('nan')):.3f} "
                f"(cov={m.get('sign_cov', 0):.2f}) | "
                f"n_finite={m.get('n_finite', 0)} "
                f"n_sdp_unb={m.get('n_sdp_unbounded', 0)} "
                f"n_lp_unb={m.get('n_lp_unbounded', 0)}",
                flush=True,
            )
        else:
            print(
                f"[{_ts()}] {model}: mse_finite={m['mse_overall']:.4f} "
                f"mae_finite={m['mae_overall']:.4f}",
                flush=True,
            )
    print(f"[{_ts()}] ═══════════════════════════════════════════════════", flush=True)

[21:00:15] ═══════════════════════════════════════════════════
[21:00:15] REGRESSION KEEP-UNBOUNDED  GAP=fixed_additive
[21:00:15] TAU_SIGN=0.01  EPS_GAP=0.001
[21:00:15] decision_rule=LP-biased (delta > -TAU_SIGN)
[21:00:15] unbounded rows kept with explicit targets; ignored in finite MSE
[21:00:15] ═══════════════════════════════════════════════════
[21:00:15] [features] Not found: selected_featuresregfdr.txt - using ALL
[21:00:15] MODE=pairwise | relaxations=('lp', 'sdp_prime')
[21:00:15] EPSILON_GAP=0.001 | EPSILON_METHOD=constant
[21:00:15] DELTA_TRANSFORM=none | TAU_SIGN=0.01
[21:00:15] USE_SAMPLE_WEIGHTS=True | WEIGHT_ALPHA=0.3
[21:00:15] ═══════════════════════════════════════════════════
[21:00:15] Loading training data ...
[21:00:20] [merge] features_5_10_10a_DI_synth.xlsx + instancesobj_5_10_10a.csv -> rows=10000
[21:00:20] Loading test data ...
[21:00:49] [merge] features_10_2a_DI_synth.xlsx + instancesobj_10_2a.csv -> rows=50000
[21:00:49] Train rows: 10000 | Test rows: 50